# XiangqiMaster — Colab Training (Safe / Disconnect-Resistant)

GPU training for the XiangqiMaster agents.

This version is modified from the original notebook to reduce the chance of a Colab runtime death and to make diagnosis/resume easier.

### Main changes
- **IL batch size starts at 64** instead of 512.
- **DataLoader workers start at 0** instead of 2, reducing system-RAM pressure.
- Training is staged: **small smoke test → larger test → full training**.
- RAM and T4 VRAM are monitored and logged to Google Drive.
- Training stdout/stderr is also saved to Drive.
- Checkpoints remain on Drive.
- The notebook keeps the original CLI/entrypoints and does not assume changes to your repository code.
- After a disconnect, rerun setup and use `--resume`.

> **Important:** Runtime death can be caused by system RAM OOM, GPU VRAM OOM, a native-library crash, or Colab-side termination. This notebook cannot prevent every cause, but it gives you much better protection and diagnostics.

Pipeline: **IL (Agent 2 P1) → PPO fine-tune (Agent 2 P2)**, plus **PPO self-play (Agent 1)** and **MCTS (Agent 3)**.

> Set **Runtime → Change runtime type → GPU** first.


## 0. Check GPU, RAM, and Python environment

In [ ]:
!nvidia-smi

import os, sys, psutil, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

ram = psutil.virtual_memory()
print(f"RAM total:     {ram.total/1024**3:.2f} GB")
print(f"RAM available: {ram.available/1024**3:.2f} GB")
print(f"RAM used:      {ram.used/1024**3:.2f} GB ({ram.percent:.1f}%)")


## 1. Get the code

**Option A — clone from GitHub.** Replace `REPO_URL` with your repository URL.

For a private repo, use an appropriate authenticated Git method rather than committing a token into the notebook.


In [ ]:
REPO_URL = 'https://github.com/<your-user>/xiangqimaster.git'  # <-- EDIT

import os, subprocess

if not os.path.exists("xiangqimaster/.git"):
    subprocess.run(["git", "clone", REPO_URL, "xiangqimaster"], check=True)
else:
    print("Repository already exists; skipping clone.")

%cd xiangqimaster
print("Repo:", os.getcwd())


**Option B — use a repository already stored on Google Drive.**

Skip this cell if you cloned the repo above.


In [ ]:
# Example:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/xiangqimaster


## 2. Install dependencies

In [ ]:
!pip -q install 'gymnasium>=0.29' 'stable-baselines3>=2.2' 'sb3-contrib>=2.2' \
    'numpy>=1.24' 'pandas>=2.0' 'tqdm>=4.66' psutil gdown

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


## 3. Mount Google Drive and create persistent directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/xiangqi_ckpts'
LOG_DIR  = '/content/drive/MyDrive/xiangqi_training_logs'

import os
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("checkpoints ->", CKPT_DIR)
print("logs        ->", LOG_DIR)


## 4. Get the data & build splits

The CGLemon dpxq/WXF game files live on Google Drive. Download them into `data/raw/`, then build the train/val/test splits.

The checks below prevent downloading the same folders again if the files already exist.


In [ ]:
!pip -q install gdown

import os
os.makedirs('data/raw', exist_ok=True)

# CGLemon chinese-chess-PGN — original folder IDs from the source notebook.
# Only download if the destination appears empty.

wxf_dir = 'data/raw/wxf'
dpxq_dir = 'data/raw/dpxq'

if not os.path.exists(wxf_dir) or not os.listdir(wxf_dir):
    print("Downloading WXF...")
    !gdown --folder 1NScafJyq3qVG7HO77U_4rVQJxbyRKe0O -O data/raw/wxf
else:
    print("WXF already exists; skipping download.")

if not os.path.exists(dpxq_dir) or not os.listdir(dpxq_dir):
    print("Downloading DPXQ...")
    !gdown --folder 12Js9Ld6Yixq4RA96j1PeT2QTUJ0-z0OB -O data/raw/dpxq
else:
    print("DPXQ already exists; skipping download.")

!find data/raw -maxdepth 2 -type f | head -30


In [ ]:
import os, glob

split_files = glob.glob("data/splits/*") if os.path.exists("data/splits") else []

if split_files:
    print("Existing split files found:")
    for p in split_files[:20]:
        print(" ", p)
    print("Skipping pipeline build. Delete data/splits if you intentionally want to rebuild.")
else:
    os.makedirs("data/splits", exist_ok=True)
    !python -m src.data.pipeline --raw data/raw --out data/splits --notation iccs --spot-check 500


## 5. Verify the dataset before expensive training

In [ ]:
import os, glob

print("Split files:")
for p in sorted(glob.glob("data/splits/*")):
    try:
        size_mb = os.path.getsize(p) / 1024**2
        print(f"{p:60s} {size_mb:8.2f} MB")
    except OSError:
        print(p)


## 6. Persistent memory monitor

This creates a small monitor that writes RAM and GPU memory to a CSV file on Google Drive.

The monitor is intentionally lightweight. It is useful for diagnosing the next runtime death:

- If RAM approaches 100% immediately before the crash → system RAM is the likely problem.
- If T4 VRAM approaches its limit immediately before the crash → GPU memory is the likely problem.
- If both remain comfortable and the runtime still dies → investigate native-library/driver/Colab-side termination.


In [ ]:
MONITOR_SCRIPT = os.path.join(LOG_DIR, "memory_monitor.py")
MONITOR_LOG = os.path.join(LOG_DIR, "memory_monitor.csv")

monitor_code = r'''
import csv
import os
import subprocess
import time
from datetime import datetime, timezone

LOG = os.environ.get("XIANGQI_MEMORY_LOG", "/content/drive/MyDrive/xiangqi_training_logs/memory_monitor.csv")

os.makedirs(os.path.dirname(LOG), exist_ok=True)

def gpu_memory():
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=memory.used,memory.total,utilization.gpu",
                "--format=csv,noheader,nounits"
            ],
            stderr=subprocess.DEVNULL,
            text=True,
            timeout=5
        ).strip().splitlines()

        if out:
            parts = [x.strip() for x in out[0].split(",")]
            return parts[0], parts[1], parts[2]
    except Exception:
        pass
    return "", "", ""

def ram_memory():
    try:
        import psutil
        m = psutil.virtual_memory()
        return (
            f"{m.used/1024**3:.3f}",
            f"{m.total/1024**3:.3f}",
            f"{m.percent:.1f}"
        )
    except Exception:
        return "", "", ""

header = [
    "timestamp_utc",
    "ram_used_gb",
    "ram_total_gb",
    "ram_percent",
    "gpu_used_mb",
    "gpu_total_mb",
    "gpu_util_percent"
]

need_header = not os.path.exists(LOG) or os.path.getsize(LOG) == 0

with open(LOG, "a", newline="") as f:
    writer = csv.writer(f)
    if need_header:
        writer.writerow(header)
        f.flush()

    while True:
        r_used, r_total, r_pct = ram_memory()
        g_used, g_total, g_util = gpu_memory()

        writer.writerow([
            datetime.now(timezone.utc).isoformat(),
            r_used, r_total, r_pct,
            g_used, g_total, g_util
        ])
        f.flush()

        time.sleep(5)
'''

with open(MONITOR_SCRIPT, "w") as f:
    f.write(monitor_code)

print("Monitor script:", MONITOR_SCRIPT)
print("Monitor log:", MONITOR_LOG)


In [ ]:
# Start the memory monitor in the background.
# If a previous monitor exists in this runtime, stop it first.

import subprocess, os, signal, time

# Stop an old monitor if this notebook was rerun.
subprocess.run(["pkill", "-f", "xiangqi_training_logs/memory_monitor.py"], 
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

env = os.environ.copy()
env["XIANGQI_MEMORY_LOG"] = MONITOR_LOG

monitor_proc = subprocess.Popen(
    ["python", MONITOR_SCRIPT],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Memory monitor PID:", monitor_proc.pid)
print("Logging every 5 seconds to:", MONITOR_LOG)


## 7. IL smoke test — Agent 2, Phase 1

**Do this first. Do not start the 9.3M-position run yet.**

The original notebook used `batch-size 512` and `num-workers 2`. This version starts at:

- `batch-size 64`
- `num-workers 0`
- `epochs 1`
- `limit-train 2,000`

The purpose is only to verify that the model, data loader, CUDA, checkpointing, and Drive all work without killing the runtime.


In [ ]:
IL_CKPT = f'{CKPT_DIR}/il_agent2_phase1.pt'
IL_LOG   = f'{LOG_DIR}/il_smoke_test.log'

!python -u -m src.training.il_train --splits data/splits --epochs 1 \
    --limit-train 2000 \
    --device cuda \
    --num-workers 0 \
    --batch-size 64 \
    --save-every 100 \
    --save $IL_CKPT 2>&1 | tee $IL_LOG


## 8. Inspect memory log after the smoke test

In [ ]:
import pandas as pd, os

if os.path.exists(MONITOR_LOG):
    df = pd.read_csv(MONITOR_LOG)
    print(df.tail(20).to_string(index=False))
else:
    print("Memory log not found:", MONITOR_LOG)


## 9. IL controlled scale-up

Only continue if the smoke test completed successfully and the checkpoint exists.

Recommended progression on a T4:

1. **64 / workers 0**
2. **128 / workers 0**
3. **256 / workers 0**
4. Only then consider **256 / workers 2**
5. Try **512** only if memory logs show enough headroom.

Do not jump directly to 512 after a runtime death.


In [ ]:
import os

print("Checkpoint exists:", os.path.exists(IL_CKPT))
if os.path.exists(IL_CKPT):
    print("Checkpoint size:", os.path.getsize(IL_CKPT)/1024**2, "MB")


In [ ]:
# CONTROLLED TEST: 5,000 games, batch 128, workers 0
# Run this only after the 2,000-game smoke test succeeds.

IL_LOG_128 = f'{LOG_DIR}/il_test_batch128.log'

!python -u -m src.training.il_train --splits data/splits --epochs 1 \
    --limit-train 5000 \
    --device cuda \
    --num-workers 0 \
    --batch-size 128 \
    --save-every 200 \
    --save $IL_CKPT 2>&1 | tee $IL_LOG_128


## 10. Full IL training — safer default

For the first full run, use **batch 128 and workers 0**.

This is intentionally more conservative than the original `batch-size 512 / num-workers 2`.

If this completes reliably and the memory log has substantial headroom, you can later test batch 256.

### Full run


In [ ]:
# FULL IL RUN — conservative T4 configuration
#
# IMPORTANT:
# If this is a fresh run, use this command without --resume.
# If Colab disconnected after a checkpoint was written, use the RESUME cell below.

IL_FULL_LOG = f'{LOG_DIR}/il_full_batch128.log'

!python -u -m src.training.il_train --splits data/splits --epochs 30 \
    --device cuda \
    --num-workers 0 \
    --batch-size 128 \
    --save-every 500 \
    --save $IL_CKPT 2>&1 | tee $IL_FULL_LOG


## 11. Resume IL after a Colab disconnect

If the runtime dies, reconnect the GPU runtime, rerun the setup/data cells, and then run this cell.

The checkpoint is on Google Drive, so it should survive the runtime being destroyed.

> Use the same training configuration when resuming: batch size 128 and workers 0.


In [ ]:
# RESUME IL AFTER DISCONNECT

IL_RESUME_LOG = f'{LOG_DIR}/il_resume.log'

!python -u -m src.training.il_train --splits data/splits --epochs 30 \
    --device cuda \
    --num-workers 0 \
    --batch-size 128 \
    --save-every 500 \
    --save $IL_CKPT \
    --resume 2>&1 | tee $IL_RESUME_LOG


## 12. Checkpoint and memory diagnostics

In [ ]:
import os, glob, pandas as pd

print("=== Checkpoints ===")
for p in sorted(glob.glob(os.path.join(CKPT_DIR, "*"))):
    print(f"{os.path.basename(p):50s} {os.path.getsize(p)/1024**2:10.2f} MB")

print("\n=== Recent memory samples ===")
if os.path.exists(MONITOR_LOG):
    df = pd.read_csv(MONITOR_LOG)
    print(df.tail(30).to_string(index=False))

    if not df.empty:
        try:
            print("\nPeak RAM:", df["ram_percent"].max(), "%")
        except Exception:
            pass
        try:
            print("Peak GPU:", df["gpu_used_mb"].astype(float).max(), "MB")
        except Exception:
            pass
else:
    print("No memory log found.")


## 13. PPO fine-tune — Agent 2, Phase 2

Only start PPO after the IL checkpoint is valid.

The original PPO command is retained. If PPO itself causes runtime death, use the same diagnostic approach: start with a smaller workload and inspect RAM/VRAM.



In [ ]:
# PPO Agent 2 — IL initialization
#
# Start with a shorter run first if the runtime has been unstable.

PPO2_LOG = f'{LOG_DIR}/ppo_agent2.log'

!python -u -m src.training.ppo_train --timesteps 500000 \
    --il-checkpoint $IL_CKPT \
    --save $CKPT_DIR/ppo_agent2 2>&1 | tee $PPO2_LOG


## 14. PPO self-play — Agent 1

Independent of the IL checkpoint — random-init self-play.

Because PPO can also be long-running, keep its checkpoints on Drive and inspect the memory log if the runtime dies.


In [ ]:
PPO1_LOG = f'{LOG_DIR}/ppo_agent1.log'

!python -u -m src.training.ppo_train --timesteps 500000 \
    --save $CKPT_DIR/ppo_agent1 2>&1 | tee $PPO1_LOG


## 15. MCTS self-play — Agent 3

The original notebook notes that this is the most compute-heavy agent (AlphaZero-style).

See `docs/07-TRAINING.md §4` for the loop and hyperparameters. Keep all generated checkpoints/logs on Drive.

The entrypoint is intentionally not invented here because the original notebook did not specify a finalized MCTS training CLI.


## 16. Run experiments

With trained checkpoints on Drive, run the experiment scripts already coded in the repository.


In [ ]:
# Example — adjust to each experiment's CLI / entrypoint.
# !python -m experiments.exp1_main_comparison

!ls experiments/ 2>/dev/null || echo 'see experiments/ in the repo'


---
## Disconnect survival checklist

### Before training
- [ ] Runtime is **GPU**
- [ ] `nvidia-smi` shows the T4
- [ ] Google Drive is mounted
- [ ] `CKPT_DIR` and `LOG_DIR` are on Drive
- [ ] Smoke test completed
- [ ] Checkpoint file exists

### IL default
- [ ] `batch-size 128`
- [ ] `num-workers 0`
- [ ] `save-every 500`
- [ ] checkpoint path is on Drive

### If runtime dies
1. Reconnect the runtime.
2. Rerun the setup cells.
3. Check the Drive memory log and training log.
4. Confirm the checkpoint exists.
5. Resume with the `--resume` cell.
6. Do **not** immediately jump to batch 512.

### Interpreting the memory log
- RAM near 100% before death → likely system RAM pressure/OOM.
- GPU memory near T4 capacity before death → likely VRAM/CUDA pressure.
- Both remain well below limits → investigate native-library crash, driver/CUDA issue, or Colab-side termination.

### Important
Colab runtime death is not always caused by your Python code. The browser message `CustomError: Runtime died` only confirms that the backend runtime disappeared; the memory/training logs are what help identify the cause.
